# Лабораторна робота №4: просторові перетворення зображень

Досліджуємо **масштабування**, **поворот**, **зсув**, **віддзеркалення**, **афінне** та **перспективне** перетворення, а також вплив **методу інтерполяції** при зміні розміру. Вхідне зображення — `satir.jpg` у корені репозиторію; усі артефакти зберігаються у `Lab_04/results/`.

## 1. Імпорт бібліотек

Використовуємо `pathlib` для крос-платформених шляхів, `numpy` для масивів, `cv2` (OpenCV) для геометричних операцій та `matplotlib.pyplot` для візуалізації та збереження порівняльних сіток.

In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

## 2. Налаштування шляхів

Поточна робоча папка (`Path.cwd()`) залежить від того, звідки запущено Jupyter: з кореня `AISIP` або з `Lab_04`. Якщо у поточній папці є `Lab_04.ipynb`, вважаємо `NOTEBOOK_DIR = cwd`; інакше — підпапка `Lab_04`. Корінь репозиторію (`ROOT`) — батьківська папка для `NOTEBOOK_DIR`.

In [2]:
_cwd = Path.cwd().resolve()
NOTEBOOK_DIR = _cwd if (_cwd / "Lab_04.ipynb").is_file() else (_cwd / "Lab_04")
ROOT = NOTEBOOK_DIR.parent.resolve()
IMAGE_PATH = ROOT / "satir.jpg"
RESULTS_DIR = (NOTEBOOK_DIR / "results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("ROOT:", ROOT)
print("IMAGE_PATH:", IMAGE_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

NOTEBOOK_DIR: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\Lab_04
ROOT: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP
IMAGE_PATH: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\satir.jpg
RESULTS_DIR: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\Lab_04\results


## 3. Читання та запис з Unicode-шляхами (Windows)

Функції `cv2.imread` / `cv2.imwrite` можуть не працювати з не-ASCII символами в шляху. Використовуємо `numpy.fromfile` + `cv2.imdecode` та `cv2.imencode` + `ndarray.tofile`.

In [3]:
def imread_color_unicode(path: Path) -> np.ndarray:
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Не вдалося прочитати (колір): {path}")
    return img


def imread_gray_unicode(path: Path) -> np.ndarray:
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Не вдалося прочитати (сірий): {path}")
    return img


def imwrite_unicode(path: Path, image: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(path.suffix, image)
    if not ok:
        raise RuntimeError(f"Не вдалося закодувати: {path}")
    buf.tofile(str(path))

## 4. Завантаження `satir.jpg`

Зберігаємо кольорову копію (BGR) та версію в градаціях сірого.

In [4]:
img_color = imread_color_unicode(IMAGE_PATH)
img_gray = imread_gray_unicode(IMAGE_PATH)

imwrite_unicode(RESULTS_DIR / "original_color.png", img_color)
imwrite_unicode(RESULTS_DIR / "original_gray.png", img_gray)

h, w = img_gray.shape[:2]
print(f"Розмір зображення: {w}×{h} пікселів")

Розмір зображення: 426×640 пікселів


## 5. Масштабування (`cv2.resize`)

- зменшення до **50%** лінійних розмірів;
- збільшення у **1.5** раза.

In [5]:
new_w_down, new_h_down = int(round(w * 0.5)), int(round(h * 0.5))
new_w_up, new_h_up = int(round(w * 1.5)), int(round(h * 1.5))

scaled_down = cv2.resize(img_color, (new_w_down, new_h_down), interpolation=cv2.INTER_AREA)
scaled_up = cv2.resize(img_color, (new_w_up, new_h_up), interpolation=cv2.INTER_LINEAR)

imwrite_unicode(RESULTS_DIR / "scaled_down.png", scaled_down)
imwrite_unicode(RESULTS_DIR / "scaled_up.png", scaled_up)

## 6. Поворот на 30° навколо центра

`cv2.getRotationMatrix2D` + `cv2.warpAffine`; розмір полотна залишаємо як у вихідного зображення.

In [6]:
center = (w / 2.0, h / 2.0)
M_rot = cv2.getRotationMatrix2D(center, angle=30.0, scale=1.0)
rotated_30 = cv2.warpAffine(img_color, M_rot, dsize=(w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
imwrite_unicode(RESULTS_DIR / "rotated_30.png", rotated_30)

## 7. Зсув на 60 px вправо та 40 px вниз

Матриця афінного перетворення \(2 \times 3\): `[[1, 0, 60], [0, 1, 40]]`.

In [7]:
M_tr = np.float32([[1, 0, 60], [0, 1, 40]])
translated = cv2.warpAffine(img_color, M_tr, dsize=(w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
imwrite_unicode(RESULTS_DIR / "translated.png", translated)

## 8. Віддзеркалення (`cv2.flip`)

Горизонтальне (`flipCode=1`) та вертикальне (`flipCode=0`).

In [8]:
flipped_h = cv2.flip(img_color, 1)
flipped_v = cv2.flip(img_color, 0)
imwrite_unicode(RESULTS_DIR / "flipped_horizontal.png", flipped_h)
imwrite_unicode(RESULTS_DIR / "flipped_vertical.png", flipped_v)

## 9. Афінне перетворення

Три вихідні та три цільові точки обчислюємо автоматично від \(w, h\): лівий верхній, правий верхній, лівий нижній кути трохи «зсуваємо» для наочної деформації.

In [9]:
src_aff = np.float32([[0, 0], [w - 1, 0], [0, h - 1]])
dst_aff = np.float32(
    [[w * 0.08, h * 0.12], [w * 0.92, h * 0.06], [w * 0.06, h * 0.90]]
)
M_aff = cv2.getAffineTransform(src_aff, dst_aff)
affine_transform = cv2.warpAffine(img_color, M_aff, dsize=(w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
imwrite_unicode(RESULTS_DIR / "affine_transform.png", affine_transform)

## 10. Перспективне перетворення

Чотири кути вихідного прямокутника відображаємо на чотири автоматично обрані точки всередині поля (легка «трапеція»).

In [10]:
src_per = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
dst_per = np.float32(
    [[w * 0.10, h * 0.08], [w * 0.90, h * 0.10], [w * 0.88, h * 0.92], [w * 0.12, h * 0.90]]
)
M_per = cv2.getPerspectiveTransform(src_per, dst_per)
perspective_transform = cv2.warpPerspective(img_color, M_per, dsize=(w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
imwrite_unicode(RESULTS_DIR / "perspective_transform.png", perspective_transform)

## 11. Порівняння інтерполяцій при збільшенні у 2 рази

`INTER_NEAREST`, `INTER_LINEAR`, `INTER_CUBIC`.

In [11]:
tw, th = int(round(w * 2)), int(round(h * 2))
interp_nearest = cv2.resize(img_color, (tw, th), interpolation=cv2.INTER_NEAREST)
interp_linear = cv2.resize(img_color, (tw, th), interpolation=cv2.INTER_LINEAR)
interp_cubic = cv2.resize(img_color, (tw, th), interpolation=cv2.INTER_CUBIC)

imwrite_unicode(RESULTS_DIR / "interpolation_nearest.png", interp_nearest)
imwrite_unicode(RESULTS_DIR / "interpolation_linear.png", interp_linear)
imwrite_unicode(RESULTS_DIR / "interpolation_cubic.png", interp_cubic)

## 12–13. Порівняльні зображення

- **`comparison.png`**: сітка **2×4** — оригінал, зменшення, збільшення, поворот, зсув, горизонтальне дзеркало, афінне, перспективне перетворення (для перегляду — RGB).
- **`interpolation_comparison.png`**: nearest / linear / cubic.

In [12]:
def bgr_to_rgb(im: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(2, 4, figsize=(14, 7))
titles = [
    "Оригінал",
    "Зменшення 50%",
    "Збільшення ×1.5",
    "Поворот 30°",
    "Зсув (+60, +40)",
    "Дзеркало гориз.",
    "Афінне",
    "Перспектива",
]
imgs = [img_color, scaled_down, scaled_up, rotated_30, translated, flipped_h, affine_transform, perspective_transform]
for ax, im, title in zip(axes.ravel(), imgs, titles):
    ax.imshow(bgr_to_rgb(im))
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)

fig2, axes2 = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, title in zip(
    axes2,
    [interp_nearest, interp_linear, interp_cubic],
    ["INTER_NEAREST", "INTER_LINEAR", "INTER_CUBIC"],
):
    ax.imshow(bgr_to_rgb(im))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
fig2.savefig(RESULTS_DIR / "interpolation_comparison.png", dpi=120, bbox_inches="tight")
plt.close(fig2)

## 14. Перевірка створених файлів

Усі очікувані PNG мають з’явитися у `RESULTS_DIR`.

In [13]:
expected = [
    "original_color.png",
    "original_gray.png",
    "scaled_down.png",
    "scaled_up.png",
    "rotated_30.png",
    "translated.png",
    "flipped_horizontal.png",
    "flipped_vertical.png",
    "affine_transform.png",
    "perspective_transform.png",
    "interpolation_nearest.png",
    "interpolation_linear.png",
    "interpolation_cubic.png",
    "comparison.png",
    "interpolation_comparison.png",
]
missing = [name for name in expected if not (RESULTS_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Відсутні файли: " + ", ".join(missing))
print("Усі результати лабораторної роботи №4 успішно створено.")

Усі результати лабораторної роботи №4 успішно створено.
